# Pipeline de VALIDACIÓN (DONREP) — hold-out para medir generalización

Copia de `mvp_pipeline_full.ipynb`. En vez de correr sobre todo el catálogo o sobre el golden set de calibración, muestrea un **conjunto de validación hold-out**: filas *distintas* de las que se usaron para calibrar umbrales/prompt (`mvp_pipeline.ipynb`), para medir qué tan bien generaliza el pipeline sin re-tunear sobre las mismas filas (evita el sobreajuste).

Usa cache y output propios (`cache_val/`, `output_val/`) para no mezclarse con la corrida de calibración (`cache/`) ni con la corrida completa (`cache_full/`). Como es una corrida nueva y aislada, **cada ref cuesta llamadas reales** — el muestreo se mantiene chico (~2/categoría) para caber en el tope diario gratis del CSE (100 queries) y es re-corrible en días sucesivos.

## Etapa 0 — Carga de datos

Librerias 

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path


In [ ]:
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

INPUT_XLSX = ROOT / "input" / "products.xlsx"
COL_REF, COL_BRAND, COL_NAME = "Ref Proveedor", "Marca", "Nombre"

df = pd.read_excel(INPUT_XLSX, dtype={COL_REF: str})
df = df[[COL_REF, COL_BRAND, COL_NAME]].copy()
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()
df = df[df[COL_REF].notna() & (df[COL_REF] != "") & (df[COL_REF] != "nan")].reset_index(drop=True)

print(f"{len(df)} productos cargados")
df.head(10)

In [ ]:
# Cache/output aislados: cache_val/ y output_val/ (ni cache/ de calibracion
# ni cache_full/ de la corrida completa). Se parchan los CACHE_DIR/LEDGER/
# IMG_DIR de cada modulo (son globals de modulo, referenciados en tiempo de
# llamada -> el parche aplica a TODAS las funciones del pipeline sin tocar
# pipeline/*.py). Correr esta celda antes que cualquier otra que use cache.
import pipeline.search as search_mod
import pipeline.prefilter as prefilter_mod
import pipeline.select as select_mod
import pipeline.fallback as fallback_mod
import pipeline.metrics as metrics_mod
import pipeline.io_excel as io_excel_mod

CACHE_ROOT = ROOT / "cache_val"
OUTPUT_ROOT = ROOT / "output_val"

search_mod.CACHE_DIR = CACHE_ROOT / "cse"
prefilter_mod.CACHE_DIR = CACHE_ROOT / "prefilter"
select_mod.CACHE_DIR = CACHE_ROOT / "select"
fallback_mod.CACHE_DIR = CACHE_ROOT / "fallback"
metrics_mod.LEDGER = CACHE_ROOT / "metrics" / "events.jsonl"
io_excel_mod.IMG_DIR = OUTPUT_ROOT / "images"

for _d in [search_mod.CACHE_DIR, prefilter_mod.CACHE_DIR, select_mod.CACHE_DIR,
           fallback_mod.CACHE_DIR, metrics_mod.LEDGER.parent, io_excel_mod.IMG_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print(f"cache -> {CACHE_ROOT}")
print(f"output -> {OUTPUT_ROOT}")

## Etapa 1 — Normalización de nombres

Es importante revisar visualmente `nombre_original` vs `nombre_limpio` / `termino_busqueda` luego de la normalizacion. De ser necesario se puede ajustar `config/abbreviations.py` .

In [ ]:
from pipeline.normalize import normalize_df

df_norm = normalize_df(df, col_nombre=COL_NAME, col_marca=COL_BRAND)

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 300)
df_norm[["nombre_original", "nombre_limpio", "termino_busqueda"]].head(10)

In [ ]:
# columnas estructuradas extraídas
df_norm[["nombre_original", "pack", "litraje", "pack_ambiguo", "viscosidad", "medida_llanta", "vehiculo_code_raw"]].head(10)

In [ ]:
df_norm[["nombre_original", "pack", "litraje", "pack_ambiguo", "viscosidad", "medida_llanta", "vehiculo_code_raw"]].tail(10)

### Filas de pack ambiguo (revisar a mano)

Tienen expresión de pack pero sin "L" explícito, así que no se pudo separar litraje de multiplicador de caja - quedaron intactas en `nombre_limpio`. ¿`4X4 UNID` es un típo de `4LX4UND`? ¿Qué significan `3X5` / `6X1` de Castrol?

In [ ]:
df_norm[df_norm["pack_ambiguo"]][["nombre_original", "nombre_limpio", "pack"]]

## Etapa 2 — Categorización de productos, con base en los nombres


In [ ]:
from pipeline.categorize import categorize_df, coverage_report
coverage_report(df_norm)          # prints per-category counts + the `otros` backlog list
cd = categorize_df(df_norm)       # adds categoria / cse_profile / presentacion columns
cd[["nombre_limpio", "categoria", "cse_profile"]].head(10)

## Etapa 3 — Búsqueda CSE

Una llamada por producto. **Rung 1 (`"{ref}" "{marca}"`, ambas comilladas)** es el default: la query mínima y probada. La respuesta cruda del CSE se cachea en `cache/cse/{ref}/rung1.json`, así que iterar el resto del pipeline **no vuelve a pagar** búsquedas.

Perfil por categoría → params de imagen: `baseline` = `imgType=photo,imgSize=large`; `white_dominant` añade `imgDominantColor=white`; `exact_brand` añade `exactTerms={marca}`. Sesgo de mercado `gl=co&hl=es`.

Los rungs 2–5 (sin comillas / `nombre_limpio` / etc.) existen en `pipeline/search.py` pero **no** se auto-escalan todavía

In [ ]:
# Conjunto de VALIDACION hold-out: NO es el golden de calibracion. Se
# muestrea aparte para medir generalizacion sin re-tunear sobre las mismas
# filas. La variable se sigue llamando `golden` para no tocar el resto del
# notebook.
#
#  - Excluye el golden-30 (groupby.head(2)) contra el que se calibro.
#  - Muestreo estratificado por categoria con semilla fija -> reproducible,
#    pero distinto del sesgo determinista de .head(2): baraja con SEED y toma
#    los primeros N por categoria (tolera categorias con < N filas).
#  - cache_val/ arranca vacio -> cada ref cuesta llamadas reales. Con
#    N_PER_CAT=2 cabe en el tope diario gratis del CSE (100) y es re-corrible.
N_PER_CAT = 2
SEED = 42

# mismas filas que uso la calibracion (mvp_pipeline.ipynb, celda del golden)
golden_calib = cd.groupby("categoria", group_keys=False).head(2)
resto = cd[~cd["Ref Proveedor"].isin(golden_calib["Ref Proveedor"])]

golden = (resto.sample(frac=1, random_state=SEED)          # baraja reproducible
          .groupby("categoria", group_keys=False)
          .head(N_PER_CAT)                                   # N por categoria
          .reset_index(drop=True))

# sanity: cero solape con el set de calibracion
assert not (set(golden["Ref Proveedor"]) & set(golden_calib["Ref Proveedor"])), \
    "el set de validacion se solapa con el de calibracion"

print(f"Set de validacion (hold-out): {len(golden)} productos, "
      f"{golden['categoria'].nunique()} categorias | "
      f"disjunto del golden-{len(golden_calib)} de calibracion | SEED={SEED}")
golden[["Ref Proveedor", "Marca", "nombre_limpio", "categoria", "cse_profile"]]

In [ ]:
# rung = cual query construir:  1 -> "ref" "marca" (default) | 2 -> ref marca | 3 -> nombre_limpio
# max_queries = tope de llamadas NUEVAS a la API. OJO: el CSE gratis da
# solo 100 queries NUEVAS por dia (una sola key en .env) -> con 281
# productos esta celda NO se completa en un solo dia. Se puede re-correr
# en dias sucesivos: lo ya buscado queda en cache_full/cse/{ref}/... y no
# se vuelve a pagar, solo se piden las que faltan hasta agotar el limite
# diario de nuevo.
from pipeline.search import search_df

resultados = search_df(golden, rung=1, max_queries=len(golden))

# Resumen: cuantos candidatos trajo cada producto y de que dominios
resumen = pd.DataFrame([
    {"ref": ref,
     "n_candidatos": len(cands),
     "dominios": ", ".join(sorted({c["displayLink"] for c in cands})[:5])}
    for ref, cands in resultados.items()
])
resumen

In [ ]:
import matplotlib.pyplot as plt

# --- Productos sin ningún candidato ---
sin_candidatos = resumen[resumen["n_candidatos"] == 0]
pct_sin = 100 * len(sin_candidatos) / len(resumen)
print(f"Productos sin candidatos: {len(sin_candidatos)} de {len(resumen)} ({pct_sin:.1f}%)")
sin_candidatos

# --- Distribución de cuántos candidatos trajo cada producto ---
distribucion = (
    resumen["n_candidatos"]
    .value_counts()
    .sort_index()
    .rename_axis("n_candidatos")
    .reset_index(name="n_productos")
)
distribucion


In [ ]:
# --- Gráfico de barras de la distribución ---
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    distribucion["n_candidatos"].astype(str),
    distribucion["n_productos"],
    color="#4C72B0",
    width=0.7,
)

ax.set_xlabel("Número de candidatos")
ax.set_ylabel("Número de productos")
ax.set_title("Distribución de candidatos por producto")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="lightgray", linewidth=0.5, alpha=0.7)
ax.set_axisbelow(True)

# etiquetas encima de cada barra
for x, y in zip(distribucion["n_candidatos"].astype(str), distribucion["n_productos"]):
    ax.text(x, y + max(distribucion["n_productos"]) * 0.01, str(y), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# Preview inline: miniaturas por producto para revisar a ojo el pool crudo
from IPython.display import HTML, display

for ref, cands in resultados.items():
    fila = golden[golden["Ref Proveedor"] == ref].iloc[0]
    print(f"{ref}  |  {fila['nombre_limpio']}  |  {fila['categoria']} "
        f"({fila['cse_profile']})  |  {len(cands)} candidatos")
    imgs = "".join(
        f'<img src="{c["thumbnailLink"]}" title="{c["displayLink"]}" '
        f'style="height:90px;margin:2px;border:1px solid #ccc">'
        for c in cands if c["thumbnailLink"]
    )
    display(HTML(imgs or "<i>(sin candidatos)</i>"))


## Etapa 4 — Pre-filtro (sin Gemini)

Sobre el pool crudo de la Etapa 3: dedupe por URL y por contenido visual (perceptual hash —> misma foto servida desde dos dominios distintos), descarta resolución baja / aspect ratio de banner / dominios en `config/domains.py`, y luego **ordena** (no descarta) por dominio prioritario + score de fondo blanco calculado del thumbnail. Los primeros `keep=6` que sobreviven se verifican con un HEAD (sin bajar la imagen completa —> Gemini la pedirá después, en memoria).

Todo el proceso escribe **solo un JSON de resultado por producto** en `cache/prefilter/{ref}.json` — no se guardan imágenes a disco, así que re-correr esta celda no vuelve a pegarle a la red.

In [ ]:
# Corre el pre-filtro sobre el pool de la Etapa 3 (golden set). use_cache=True
# por defecto en prefilter_product: re-correr esta celda no re-hace los HEAD
# ni vuelve a bajar thumbnails, sirve el JSON de cache/prefilter/{ref}.json.
from pipeline.prefilter import prefilter_product

filtrados = {ref: prefilter_product(ref, cands) for ref, cands in resultados.items()}

resumen_prefiltro = pd.DataFrame([
    {"ref": ref,
     "n_input": r["n_input"],
     "n_kept": len(r["kept"]),
     "n_dropped": len(r["dropped"])}
    for ref, r in filtrados.items()
])
resumen_prefiltro


In [ ]:
# Preview inline: candidatos sobrevivientes (con su bg_score) y por que se
# cayeron los demas -> clave para calibrar MIN_SIDE / MAX_ASPECT_RATIO /
# HASH_MAX_DISTANCE y para ir llenando config/domains.py con evidencia real.
for ref, r in filtrados.items():
    fila = golden[golden["Ref Proveedor"] == ref].iloc[0]
    print(f"{ref}  |  {fila['nombre_limpio']}  |  {fila['categoria']}  "
        f"|  {len(r['kept'])}/{r['n_input']} sobrevivieron")

    imgs = "".join(
        f'<div style="display:inline-block;text-align:center;margin:2px">'
        f'<img src="{c["thumbnailLink"]}" title="{c["displayLink"]}" '
        f'style="height:90px;border:1px solid #ccc"><br>'
        f'<small>{c["bg_score"]}</small></div>'
        for c in r["kept"] if c["thumbnailLink"]
    )
    display(HTML(imgs or "<i>(sin candidatos sobrevivientes)</i>"))

    if r["dropped"]:
        razones = pd.Series([reason for _, reason in r["dropped"]]).value_counts()
        print("  descartados:", dict(razones))


## Etapa 5 — Selección con Gemini (Vertex AI)

Una llamada por producto: las candidatas que sobrevivieron el pre-filtro (columna `kept` de la Etapa 4), numeradas, más un prompt que pide JSON estructurado (`response_mime_type="application/json"` + `response_schema`, mismos campos que el plan: `evaluaciones`, `seleccion`, `ranking`, `confianza`, `comentario`). Gemini evalúa criterios eliminatorios (producto correcto, marca) y de calidad (resolución, fondo, estado, empaque, limpieza), y ELIGE la mejor (se plantea un código a utilizar sobre elección en los casos borde).

Modelo por defecto: `gemini-2.5-flash` (barato, discriminación no generación). `MODEL_PRO` está declarado en `pipeline/select.py` para el ruteo a dos niveles (confianza baja → Pro) pero ese ruteo automático **no** está implementado todavía —> se pasa `model=` explícito si se quiere probar.

Reglas post-Gemini ya aplicadas dentro de `select_product`:
- `seleccion` fuera de rango → se resuelve al siguiente número del propio `ranking`, sin nueva llamada.
- marca de la imagen elegida = `no_verificable` → flag `revisar_marca`.
- `seleccion=null` o `confianza=baja` → flag `necesita_fallback` (la Etapa 5b la re-dispara automáticamente por los rungs 2–3).

Cache en `cache/select/{ref}.json` —> re-correr esta celda no vuelve a pagar la llamada a Gemini. Requiere `credentials.json` + `GOOGLE_CLOUD_PROJECT`/`GOOGLE_CLOUD_LOCATION` en `.env` (ya configurados).

In [ ]:
# max_calls = tope de llamadas NUEVAS a Gemini. Vertex AI no tiene el
# limite gratis diario del CSE, pero SI se cobra por llamada -> correr esto
# sobre el catalogo completo paga precio de lista por cada producto nuevo
# (~281 la primera vez). Bajar max_calls si se quiere probar en tandas.
from pipeline.select import select_df, MODEL_FLASH

seleccionados = select_df(golden, filtrados, model=MODEL_FLASH, max_calls=len(golden))

resumen_seleccion = pd.DataFrame([
    {"ref": ref,
    "confianza": r["confianza"],
    "flags": ", ".join(r["flags"]) or "-",
    "comentario": r["comentario"]}
    for ref, r in seleccionados.items()
])
resumen_seleccion

In [ ]:
# Preview inline: candidato elegido (borde verde) + el resto del pool con la
# razon de Gemini si lo descarto. Flags a revisar a mano:
#   revisar_marca      -> la marca de la elegida no se pudo verificar
#   necesita_fallback  -> seleccion=null o confianza baja (siguiente escalon
#                          de busqueda de la Etapa 3, todavia manual)
# Se salta lo que ni siquiera se ha buscado en la Etapa 3 (ref no esta en
# `resultados`): eso no es un pool vacio real, es que search_df/escalate_df
# todavia no le tocaba (limite diario del CSE) -> no aporta nada revisarlo
# a mano todavia, solo ruido.
n_no_buscado = 0
for ref, r in seleccionados.items():
    if ref not in resultados:
        n_no_buscado += 1
        continue

    fila = golden[golden["Ref Proveedor"] == ref].iloc[0]
    sel = r["seleccion"]
    kept = filtrados.get(ref, {}).get("kept", [])
    evals = {e["imagen"]: e for e in r["evaluaciones"]}

    print(f"{ref}  |  {fila['nombre_limpio']}  |  {fila['categoria']}  "
        f"|  confianza={r['confianza']}  |  flags={', '.join(r['flags']) or '-'}")
    print(f"  {r['comentario']}")

    imgs = ""
    for i, c in enumerate(kept, start=1):
        is_sel = sel is not None and c.get("link") == sel.get("link")
        e = evals.get(i, {})
        label = "ELEGIDA" if is_sel else (e.get("razon", "") if e.get("descartada") else "")
        imgs += (
            f'<div style="display:inline-block;text-align:center;margin:3px;max-width:110px">'
            f'<img src="{c["thumbnailLink"]}" title="{c["displayLink"]}" '
            f'style="height:90px;border:{"3px solid #2a2" if is_sel else "1px solid #ccc"}"><br>'
            f'<small>{label}</small></div>'
        )
    display(HTML(imgs or "<i>(sin candidatos evaluados)</i>"))

if n_no_buscado:
    print(f"\n({n_no_buscado} productos omitidos: aun no buscados en Etapa 3, "
        f"pendientes del limite diario del CSE)")


## Etapa 5b — Fallback (rungs 2–3)

Re-ataca SOLO los productos que salieron de la Etapa 5 con `sin_imagen` / `necesita_fallback`: baja por la escalera de búsqueda (`rung 2` = ref+marca sin comillas, `rung 3` = `nombre_limpio`), fusiona los candidatos nuevos con el pool que ya había (dedupe por link), **excluye** las imágenes que un Gemini anterior condenó por criterio eliminatorio (producto incorrecto / marca incorrecta —> no se re-pagan ni ocupan los 6 cupos del pre-filtro; las descartadas solo por calidad o con marca `no_verificable` sí se quedan, pueden ganar contra un pool más débil), re-corre pre-filtro + selección Gemini, y se detiene en el primer rung que resuelva. `max_queries` / `max_calls` protegen los presupuestos igual que en las Etapas 3 y 5.

El resultado escalado queda en `cache/fallback/{ref}.json` —> re-correr esta celda no vuelve a pagar. `escalate_df` MUTA `resultados` / `filtrados` / `seleccionados` en su lugar, así que las celdas de la Etapa 6 exportan el pipeline ya rescatado sin cambios. El log retornado (una fila por ref+rung intentado) es la métrica de cuánto rescató cada escalón extra.

In [ ]:
# Escalado automatico para los productos flagged tras la Etapa 5. El log
# muestra que rung rescato que producto (o donde sigue perdido). Solo
# ataca los que quedaron sin_imagen/necesita_fallback, asi que un tope
# igual al tamano del catalogo no sobre-gasta.
from pipeline.fallback import escalate_df

log_fallback = escalate_df(golden, resultados, filtrados, seleccionados,
                           max_queries=len(golden), max_calls=len(golden))
log_fallback

## Métricas — cobertura y costo

Dos vistas sobre el mismo run:

- **Cobertura** (`selection_report`): embudo por producto (candidatos CSE → sobreviven pre-filtro → evaluadas por Gemini → seleccionada) + tasa de éxito, confianza y flags. La línea "sin imagen por etapa" dice DÓNDE se pierde cada producto —> si el pool CSE está vacío, el fix es la búsqueda (rungs/perfiles); si Gemini descarta todo, el fix es el prompt o el pre-filtro.
- **Costo** (`cost_report` / `retry_report`): leídos del ledger `cache/metrics/events.jsonl`, donde cada llamada NUEVA (no cacheada) a CSE/Gemini y cada reintento escriben una línea. Los cache-hits no cuestan y no se registran. Los tokens de Gemini vienen de `usage_metadata` (el thinking se factura como salida) y quedan también dentro de `cache/select/{ref}.json` (`usage`, `modelo`, `intentos`).

Nota: los `cache/select/{ref}.json` creados ANTES de esta instrumentación no traen tokens; para backfillear, borrar `cache/select/` y re-correr la Etapa 5 (se re-paga la llamada).

In [ ]:
# Embudo de cobertura por producto + resumen agregado (tasa de exito,
# confianza, flags, y en que etapa se pierden los productos sin imagen).
from pipeline.metrics import selection_report

por_ref, resumen = selection_report(golden, resultados, filtrados, seleccionados)
por_ref

In [ ]:
# Gasto real acumulado del ledger (queries CSE facturables por dia + tokens
# Gemini a precios de lista) y reintentos por error transitorio.
from pipeline.metrics import cost_report, retry_report

display(cost_report())
retry_report()

## Etapa 6 — Salida a Excel

Copia el catálogo original y le agrega las columnas del plan: `imagen_url`, `imagen_local`, `categoria`, `termino_busqueda`, `perfil_cse_usado`, `intento_cse`, `query_cse`, `pasadas_gemini`, `reintentos_api_gemini`, `confianza`, `flags`, `razon_gemini`. La imagen que eligió Gemini se descarga a disco en `output/images/{ref}.jpg` y se agrega como miniatura real en una columna nueva `miniatura` del `.xlsx` final, para revisar el catálogo completo sin abrir una sola URL.

Nota de flags: el plan menciona `fallback_usado`, pero el pipeline emite `necesita_fallback`; la Etapa 5b ya re-dispara automáticamente los rungs 2–3, así que un producto que llega aquí con ese flag agotó la escalera —> la columna `flags` refleja tal cual lo que trae `seleccionados`, sin renombrar.

`output/` está en `.gitignore`

In [ ]:
# download_images=True baja a disco la imagen elegida por producto
# (output_full/images/{ref}.jpg), cacheada -> re-correr no vuelve a descargar.
from pipeline.io_excel import build_output_df

out_df = build_output_df(golden, seleccionados)
out_df[["Ref Proveedor", "Marca", "categoria", "confianza", "flags", "intento_cse", "query_cse", "pasadas_gemini", "reintentos_api_gemini", "imagen_url", "imagen_local"]]

In [ ]:
# Escribe el xlsx y coloca la miniatura de cada imagen_local en una
# columna nueva ("miniatura") -> abrir output_val/catalogo_validacion.xlsx
# para revisar el set de validacion sin salir de Excel.
from pipeline.io_excel import write_excel

OUTPUT_XLSX = OUTPUT_ROOT / "catalogo_validacion.xlsx"
write_excel(out_df, OUTPUT_XLSX)

## Métricas - gráficos (embudo y promedios)

Mismos datos que ya imprime `selection_report` / `cost_report` en la sección anterior (`por_ref`, `resumen`), pero en gráficos para verlos de un vistazo:

- **Embudo de cobertura**: de los productos totales, cuántos llegan vivos a cada etapa (candidatos CSE → sobreviven pre-filtro → evaluadas por Gemini → imagen seleccionada). Muestra DÓNDE se cae cada producto (ej. pasar de 10 a 0 candidatos), no solo cuántos se pierden en total.
- **Promedio de imágenes por etapa**: cuántas imágenes en promedio llega a manejar el pipeline en cada etapa — candidatos CSE crudos, cuántas sobreviven el pre-filtro (= las que se le mandan a Gemini), cuántas evalúa Gemini realmente. Incluye también la retención condicional (de los productos que sí llegaron a una etapa, qué % de imágenes sobrevive a la siguiente) para aislar el efecto de cada filtro del efecto de "no tenía nada que evaluar".
- **Por qué se pierden los productos sin imagen, confianza y flags**: el desglose de `resumen['perdidos_...']`, `confianza` y `flags` en barras, para comparar de un vistazo qué tan seguido pasa cada cosa.

Requiere haber corrido la celda de `selection_report` (variables `por_ref` y `resumen`) más arriba.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Paleta ordinal (un solo hue, mas oscuro = etapa mas avanzada del embudo)
FUNNEL_COLORS = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]

stages = [
    ("Productos", int(resumen["productos"])),
    ("Con candidatos CSE", int((por_ref["n_cse"] > 0).sum())),
    ("Sobreviven prefiltro", int((por_ref["n_prefiltro"] > 0).sum())),
    ("Evaluadas por Gemini", int((por_ref["n_evaluadas"] > 0).sum())),
    ("Imagen seleccionada", int(resumen["con_imagen"])),
]
labels = [s[0] for s in stages]
values = [s[1] for s in stages]
total = values[0]

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(labels))[::-1]  # primera etapa arriba

ax.barh(y, values, color=FUNNEL_COLORS, height=0.6)

prev_values = [None] + values[:-1]
for yi, v, prev in zip(y, values, prev_values):
    pct_total = f"{100 * v / total:.0f}% del total" if total else "-"
    retencion = f", {100 * v / prev:.0f}% vs etapa anterior" if prev else ""
    ax.text(v + total * 0.015, yi, f"{v}  ({pct_total}{retencion})", va="center", fontsize=9)

ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlim(0, total * 1.35 if total else 1)
ax.set_xlabel("Número de productos")
ax.set_title("Embudo de cobertura: dónde se pierden los productos")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)
ax.grid(axis="x", color="#e1e0d9", linewidth=0.5, alpha=0.7)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Promedio de candidatos que llegan a cada etapa, sobre TODOS los productos
# (incluyendo los que se quedan en 0 -> refleja la carga real del pipeline,
# no solo la de los que "tuvieron suerte")
avg_stage = {
    "CSE\n(candidatos crudos)": por_ref["n_cse"].mean(),
    "Prefiltro\n(enviadas a Gemini)": por_ref["n_prefiltro"].mean(),
    "Evaluadas\npor Gemini": por_ref["n_evaluadas"].mean(),
}

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(avg_stage.keys(), avg_stage.values(),
            color=["#5598e7", "#2a78d6", "#1c5cab"], width=0.55)

for bar, v in zip(bars, avg_stage.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.05, f"{v:.1f}",
            ha="center", va="bottom", fontsize=9)

ax.set_ylabel("Promedio de imágenes por producto")
ax.set_title(f"Promedio de imágenes por etapa (sobre los {len(por_ref)} productos)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.5, alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

# Retención condicional: de los productos que SÍ llegaron vivos a una etapa,
# qué tanto sobrevive a la siguiente -> aísla el efecto de cada filtro del
# efecto de "no tenía nada que evaluar en primer lugar"
con_cse = por_ref[por_ref["n_cse"] > 0]
con_prefiltro = por_ref[por_ref["n_prefiltro"] > 0]

print(f"Promedio candidatos CSE (todos los productos):        {por_ref['n_cse'].mean():.1f}")
print(f"Promedio candidatos CSE (solo con pool no vacío):      {con_cse['n_cse'].mean():.1f}")
print(f"Promedio sobrevivientes prefiltro (mismo grupo):       {con_cse['n_prefiltro'].mean():.1f}"
    f"  -> retiene {100 * con_cse['n_prefiltro'].mean() / con_cse['n_cse'].mean():.0f}% del pool CSE")
print(f"Promedio evaluadas por Gemini (solo con prefiltro no vacío): {con_prefiltro['n_evaluadas'].mean():.1f}"
    f"  -> retiene {100 * con_prefiltro['n_evaluadas'].mean() / con_prefiltro['n_prefiltro'].mean():.0f}% del pool prefiltro")

In [ ]:
# Por que se pierden los productos sin imagen (en que etapa se quedaron sin
# nada), que tan confiada estuvo la seleccion de Gemini, y que tan seguido
# aparece cada flag a revisar a mano.
razones_perdida = {
    "Pool CSE\nvacío": resumen["perdidos_pool_cse_vacio"],
    "Prefiltro\nvacío": resumen["perdidos_prefiltro_vacio"],
    "Gemini\ndescartó todo": resumen["perdidos_gemini_descarto"],
}

conf_order = ["alta", "media", "baja"]
conf_colors = ["#0ca30c", "#fab219", "#d03b3b"]  # paleta de estado: buena/alerta/critica
conf_counts = por_ref["confianza"].value_counts()
conf_vals = [int(conf_counts.get(c, 0)) for c in conf_order]

flag_order = ["sin_imagen", "necesita_fallback", "revisar_marca", "descarga_thumbnail"]
flag_colors = ["#e34948", "#eda100", "#e87ba4", "#4a3aa7"]  # slots categoricos fijos
flag_vals = [resumen["flags"].get(f, 0) for f in flag_order]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

def _bar_panel(ax, x_labels, values, colors, title):
    bars = ax.bar(x_labels, values, color=colors, width=0.6)
    ymax = max(values) if values and max(values) else 1
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + ymax * 0.02, str(v),
                ha="center", va="bottom", fontsize=9)
    ax.set_title(title)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="#e1e0d9", linewidth=0.5, alpha=0.7)
    ax.set_axisbelow(True)

_bar_panel(axes[0], list(razones_perdida.keys()), list(razones_perdida.values()), FUNNEL_COLORS[1::2],
        f"Sin imagen: {resumen['sin_imagen']} de {resumen['productos']}, ¿en qué etapa?")
_bar_panel(axes[1], conf_order, conf_vals, conf_colors, "Confianza de Gemini en la selección")
_bar_panel(axes[2], flag_order, flag_vals, flag_colors, "Flags a revisar a mano")
axes[2].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()